Leer copias básicas y obtener los datos.

In [1]:
import pandas as pd
import glob
from pdfminer.pdfparser import PDFParser
from pdfminer.pdfdocument import PDFDocument
from pdfminer.pdftypes import resolve1, PDFObjRef

In [11]:
dirpath='C:/Users/mvgarcia/Downloads/ComiteEmpresa/cb/2024/'
filemask='*_CB.PDF'
pathmask=dirpath+filemask

In [12]:
def load_form(filename): ## Load the whole document in memory
    """Load PDF form contents"""
    with open(filename, 'rb') as file:
        parser = PDFParser(file)
        doc = PDFDocument(parser)
        return [load_fields(resolve1(f)) for f in resolve1(doc.catalog['AcroForm'])['Fields']]

In [13]:
def load_fields(field): ## Load all fields as hierarchical pairs of key-value
    """Recursively load from fields"""
    form = field.get('Kids',None)
    if form:
        return [load_fields(resolve1(f)) for f in form]
    else:
        # Some field types (e.g. signatures) need extra resolving
        return (field.get('T').decode('utf-16'), resolve1(field.get('V')))

In [14]:
def text_get(data,key): ## Once you find the field-tags you need, decode them to suit your needs
    if isinstance(data, bytes):
        text = data.decode(encoding='cp1252')
        if key.startswith('TOTAL'):
            text = text.strip()
        return text
    else:
        if data is None:
            return ''
        else:
            return data

In [15]:
def busqueda_jerarquica(datos, datos_solicitados): ## Navigate the tree to find tuples (key,value). Debug this to find out your keys
    for i in datos:
        if isinstance(i,list):
            busqueda_jerarquica(i,datos_solicitados)
        else:
            busqueda_plana(i, datos_solicitados)
            if not(None in datos_solicitados.values()):
                print('Listo')
                return

In [16]:
def busqueda_plana(datos,datos_solicitados): ## Once you find tuples, check if they are those you need
    if isinstance(datos,tuple):
        if datos[0] in datos_solicitados.keys():
            datos_solicitados[datos[0]]=text_get(datos[1],datos[0])
            print(f'({datos[0]},{datos[1]}) --> <{datos_solicitados[datos[0]]}>')
            if not(None in datos_solicitados.values()):
                return
    else:
        prinf(f"I don't know what this is: <{datos}> of class {type(datos)}")

In [17]:
def scrap_cb(pathname):
    form = load_form(pathname)
    datos_solicitados = {'RAZONSOC[0]':None, 'DODONA[1]':None, 'NIF[1]':None, 
                         'CATEGORIA[0]':None, 'FECHA[0]':None,'TOTAL[0]':None}
    datos_solicitados.keys()
    busqueda_jerarquica(form,datos_solicitados)
    return datos_solicitados

In [18]:
df = pd.DataFrame(columns=list(['Sociedad','Nombre','DNI','Categoria','Fecha','Retribucion']))
files = [f for f in glob.glob(pathmask)]
for pathname in files:
    print(f'get data from {pathname}')
    datos=scrap_cb(pathname)
    df.loc[len(df)] = datos.values()
df[['DNI', 'Fecha', 'Retribucion', 'Categoria', 'Sociedad', 'Nombre']].to_csv(dirpath+'Registro_raw.csv', index = False)

get data from C:/Users/mvgarcia/Downloads/ComiteEmpresa/cb/2024\AARON CASTRO VILA_CB.PDF
(RAZONSOC[0],b'INDRA SOLUCIONES TI') --> <INDRA SOLUCIONES TI>
(DODONA[1],b'Aar\xf3n Castro Vila') --> <Aarón Castro Vila>
(NIF[1],None) --> <>
(CATEGORIA[0],b'Area 3 Grupo E Nivel 1') --> <Area 3 Grupo E Nivel 1>
(FECHA[0],b'07.10.2024') --> <07.10.2024>
(TOTAL[0],b'          18.500,00') --> <18.500,00>
Listo
get data from C:/Users/mvgarcia/Downloads/ComiteEmpresa/cb/2024\ALEJANDRO ESTRAMIL REYES_CB.PDF
(RAZONSOC[0],b'INDRA SOLUCIONES TI') --> <INDRA SOLUCIONES TI>
(DODONA[1],b'ALEJANDRO ESTRAMIL REYES') --> <ALEJANDRO ESTRAMIL REYES>
(NIF[1],None) --> <>
(CATEGORIA[0],b'Area 3 Grupo E Nivel 1') --> <Area 3 Grupo E Nivel 1>
(FECHA[0],b'14.02.2024') --> <14.02.2024>
(TOTAL[0],b'          18.500,00') --> <18.500,00>
Listo
get data from C:/Users/mvgarcia/Downloads/ComiteEmpresa/cb/2024\ANDREA ALVITE LOPEZ_CB.PDF
(RAZONSOC[0],b'INDRA SOLUCIONES TI') --> <INDRA SOLUCIONES TI>
(DODONA[1],b'Andrea Alvite